# Assignment: K-Means and K-NN on Iris Dataset

**Course:** Machine Learning - 2  
**Date:** 2026-04-13  
**Author:** Rudra Patel
**Roll No. :** 2024SEPVUGP0037

## Overview
This notebook implements **K-Means clustering** and **K-Nearest Neighbors (K-NN)** **from scratch** using NumPy for core computations. It includes:
1. Iris data loading and inspection
2. Exploratory data analysis (EDA)
3. Custom vectorized K-Means plus Elbow method
4. Ground-truth vs. cluster visualization
5. Custom vectorized K-NN plus test-set evaluation
6. Conceptual answers, conclusion, and references

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load and Inspect the Iris Dataset
The Iris dataset is loaded using `sklearn.datasets.load_iris`. We inspect its shape, feature names, target names, and summary statistics before modeling.

In [2]:
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y
df["species"] = pd.Categorical.from_codes(y, target_names)

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Feature names: {feature_names}")
print(f"Target names: {target_names}")
df.head()

Feature matrix shape: (150, 4)
Target vector shape: (150,)
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target names: ['setosa' 'versicolor' 'virginica']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [3]:
df.info()
df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   sepal length (cm)  150 non-null    float64 
 1   sepal width (cm)   150 non-null    float64 
 2   petal length (cm)  150 non-null    float64 
 3   petal width (cm)   150 non-null    float64 
 4   target             150 non-null    int64   
 5   species            150 non-null    category
dtypes: category(1), float64(4), int64(1)
memory usage: 6.3 KB


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
count,150.000000,150.000000,150.000000,150.000000,150.000000,150
unique,NaN,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,NaN,setosa
freq,NaN,NaN,NaN,NaN,NaN,50
mean,5.843333,3.057333,3.758000,1.199333,1.000000,NaN
std,0.828066,0.435866,1.765298,0.762238,0.819232,NaN
min,4.300000,2.000000,1.000000,0.100000,0.000000,NaN
25%,5.100000,2.800000,1.600000,0.300000,0.000000,NaN
50%,5.800000,3.000000,4.350000,1.300000,1.000000,NaN
75%,6.400000,3.300000,5.100000,1.800000,2.000000,NaN


## 2. Exploratory Data Analysis (EDA)
We visualize pairwise feature relationships and compute class-wise means and covariance matrices to understand separability and overlap among species.

In [4]:
pairplot_df = df.drop(columns=["target"]).copy()
g = sns.pairplot(
    pairplot_df,
    hue="species",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.8, "s": 35}
)
g.fig.suptitle("Iris Pairplot by Species", y=1.02)
plt.show()

In [5]:
class_ids = np.unique(y)
class_means = np.vstack([X[y == class_id].mean(axis=0) for class_id in class_ids])
means_df = pd.DataFrame(class_means, index=target_names, columns=feature_names)

print("Class-wise means:")
display(means_df.round(3))

print("\nClass-wise covariance matrices:")
for class_id, class_name in enumerate(target_names):
    cov_matrix = np.cov(X[y == class_id].T)
    cov_df = pd.DataFrame(cov_matrix, index=feature_names, columns=feature_names)
    print(f"\n{class_name}:")
    display(cov_df.round(3))

Class-wise means:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
setosa,5.006,3.428,1.462,0.246
versicolor,5.936,2.770,4.260,1.326
virginica,6.588,2.974,5.552,2.026



Class-wise covariance matrices:

setosa:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
sepal length (cm),0.124,0.099,0.016,0.010
sepal width (cm),0.099,0.144,0.012,0.009
petal length (cm),0.016,0.012,0.030,0.006
petal width (cm),0.010,0.009,0.006,0.011



versicolor:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
sepal length (cm),0.266,0.085,0.183,0.056
sepal width (cm),0.085,0.098,0.083,0.041
petal length (cm),0.183,0.083,0.221,0.073
petal width (cm),0.056,0.041,0.073,0.039



virginica:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
sepal length (cm),0.404,0.094,0.303,0.049
sepal width (cm),0.094,0.104,0.071,0.048
petal length (cm),0.303,0.071,0.305,0.049
petal width (cm),0.049,0.048,0.049,0.075


### EDA Observations
- Setosa is clearly separable from the other two classes, especially in petal measurements.
- Versicolor and Virginica overlap in some feature pairs, so boundary-based methods may confuse these classes.
- Class means and covariances confirm that spread and central tendency differ by species.

## 3. K-Means Implementation (From Scratch)
In K-Means, the integer $k$ is the **number of clusters** to discover in unlabeled data.

Algorithm steps:
1. Initialize $k$ centroids (here with k-means++ style initialization).
2. Assign each sample to its nearest centroid (vectorized distance matrix).
3. Update each centroid to the mean of points assigned to it.
4. Repeat until centroid movement is below tolerance or max iterations is reached.

Per iteration, the dominant cost is distance computation: $O(nkd)$ where $n$ is samples, $k$ clusters, and $d$ features.

In [6]:
def standardize_numpy(X):
    """
    Standardize each feature column to zero mean and unit variance.

    Parameters
    ----------
    X : np.ndarray of shape (n_samples, n_features)

    Returns
    -------
    X_scaled : np.ndarray
    mean : np.ndarray
    std : np.ndarray
    """
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    X_scaled = (X - mean) / std
    return X_scaled, mean, std


def init_centroids_kmeanspp(X, k, rng):
    """
    Initialize centroids with a k-means++ style strategy.

    Parameters
    ----------
    X : np.ndarray of shape (n_samples, n_features)
    k : int
    rng : np.random.Generator

    Returns
    -------
    centroids : np.ndarray of shape (k, n_features)
    """
    n_samples, n_features = X.shape
    if k < 1 or k > n_samples:
        raise ValueError("k must be in [1, n_samples].")

    centroids = np.empty((k, n_features), dtype=float)
    first_idx = rng.integers(0, n_samples)
    centroids[0] = X[first_idx]

    closest_sq = np.sum((X - centroids[0]) ** 2, axis=1)
    for c in range(1, k):
        probs = closest_sq / closest_sq.sum()
        next_idx = rng.choice(n_samples, p=probs)
        centroids[c] = X[next_idx]

        new_sq = np.sum((X - centroids[c]) ** 2, axis=1)
        closest_sq = np.minimum(closest_sq, new_sq)

    return centroids


def kmeans(X, k, max_iters=100, tol=1e-4, random_state=None):
    """
    Run K-Means clustering using vectorized NumPy operations.

    Parameters
    ----------
    X : np.ndarray of shape (n_samples, n_features)
    k : int
        Number of clusters.
    max_iters : int, default=100
    tol : float, default=1e-4
        Stop when max centroid movement is smaller than tol.
    random_state : int or None

    Returns
    -------
    centroids : np.ndarray of shape (k, n_features)
    labels : np.ndarray of shape (n_samples,)
    inertia : float
        Sum of squared distances from each point to its assigned centroid.
    n_iters : int
        Number of iterations run.
    """
    X = np.asarray(X, dtype=float)
    n_samples, n_features = X.shape
    if k < 1 or k > n_samples:
        raise ValueError("k must be in [1, n_samples].")

    rng = np.random.default_rng(random_state)
    centroids = init_centroids_kmeanspp(X, k, rng)

    for iteration in range(1, max_iters + 1):
        # Vectorized squared Euclidean distances: shape (n_samples, k).
        distances = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2) ** 2
        labels = np.argmin(distances, axis=1)

        new_centroids = np.empty((k, n_features), dtype=float)
        for cluster_id in range(k):
            cluster_points = X[labels == cluster_id]
            if cluster_points.size == 0:
                # Re-seed empty cluster using the point with largest current residual error.
                farthest_idx = np.argmax(np.min(distances, axis=1))
                new_centroids[cluster_id] = X[farthest_idx]
            else:
                new_centroids[cluster_id] = cluster_points.mean(axis=0)

        centroid_shift = np.linalg.norm(new_centroids - centroids, axis=1).max()
        centroids = new_centroids

        if centroid_shift < tol:
            break

    final_distances = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2) ** 2
    labels = np.argmin(final_distances, axis=1)
    inertia = np.sum((X - centroids[labels]) ** 2)

    return centroids, labels, inertia, iteration

## 4. Determining $k$ with the Elbow Method
We run K-Means for $k=1$ to $10$, compute inertia (within-cluster sum of squares), and pick the elbow point where additional clusters produce diminishing returns. For Iris, a reasonable choice is typically $k=3$.

In [7]:
X_scaled, X_mean, X_std = standardize_numpy(X)

k_vals = np.arange(1, 11)
inertias = []

for k in k_vals:
    _, _, inertia_k, _ = kmeans(
        X_scaled,
        k=k,
        max_iters=200,
        tol=1e-5,
        random_state=42
    )
    inertias.append(inertia_k)

selected_k = 3

plt.figure(figsize=(8, 5))
plt.plot(k_vals, inertias, marker="o", linewidth=2)
plt.axvline(selected_k, color="red", linestyle="--", label=f"Chosen k = {selected_k}")
plt.scatter([selected_k], [inertias[selected_k - 1]], color="red", zorder=5)
plt.title("Elbow Method for K-Means on Iris")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.xticks(k_vals)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Selected k based on elbow inspection: {selected_k}")

Selected k based on elbow inspection: 3


## 5. Apply K-Means with Chosen $k$
We now run the custom implementation with $k=3$ on the standardized Iris feature matrix and inspect centroids, labels, and final inertia.

In [8]:
centroids_scaled, kmeans_labels, final_inertia, n_iters = kmeans(
    X_scaled,
    k=selected_k,
    max_iters=200,
    tol=1e-5,
    random_state=42
)

# Convert centroids back to original units for interpretability.
centroids_original = centroids_scaled * X_std + X_mean

print(f"Converged in {n_iters} iterations")
print(f"Final inertia: {final_inertia:.4f}")

centroids_df = pd.DataFrame(centroids_original, columns=feature_names)
centroids_df.index = [f"Cluster {i}" for i in range(selected_k)]
display(centroids_df.round(3))

cluster_counts = pd.Series(kmeans_labels, name="cluster").value_counts().sort_index()
print("Cluster membership counts:")
display(cluster_counts)

Converged in 6 iterations
Final inertia: 139.8205


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
Cluster 0,5.006,3.428,1.462,0.246
Cluster 1,6.781,3.096,5.511,1.972
Cluster 2,5.802,2.674,4.370,1.413


Cluster membership counts:


cluster
0    50
1    47
2    53
Name: count, dtype: int64

## 6. Visualization: Ground Truth vs. K-Means Clusters
The following side-by-side scatter plots use the same feature axes (sepal length vs. sepal width) to compare true species labels with discovered K-Means clusters.

In [9]:
x_idx, y_idx = 0, 1  # sepal length, sepal width

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

for class_id, class_name in enumerate(target_names):
    mask = y == class_id
    axes[0].scatter(
        X[mask, x_idx],
        X[mask, y_idx],
        s=45,
        alpha=0.85,
        label=class_name
    )
axes[0].set_title("Ground Truth Species")
axes[0].set_xlabel(feature_names[x_idx])
axes[0].set_ylabel(feature_names[y_idx])
axes[0].legend()

for cluster_id in range(selected_k):
    mask = kmeans_labels == cluster_id
    axes[1].scatter(
        X[mask, x_idx],
        X[mask, y_idx],
        s=45,
        alpha=0.85,
        label=f"Cluster {cluster_id}"
    )

axes[1].scatter(
    centroids_original[:, x_idx],
    centroids_original[:, y_idx],
    marker="X",
    s=220,
    c="black",
    label="Centroids"
)
axes[1].set_title("K-Means Clusters (k=3)")
axes[1].set_xlabel(feature_names[x_idx])
axes[1].set_ylabel(feature_names[y_idx])
axes[1].legend()

plt.tight_layout()
plt.show()

In this 2D projection, Setosa remains well-separated, while Versicolor and Virginica partially overlap. K-Means minimizes within-cluster distance in the full feature space, but projected views can still show overlap. Cluster identities are also unlabeled, so a perfect one-to-one visual match with species names is not guaranteed.

## 7. K-Nearest Neighbors (K-NN) Implementation (From Scratch)
In K-NN, the integer $k$ is the **number of nearest training neighbors** used to vote for each test sample's class.

Supervised workflow:
1. Store labeled training data with `fit`.
2. Compute distances from each test sample to all training samples (vectorized).
3. Select the $k$ nearest neighbors.
4. Predict via majority vote.

For one prediction, distance computation costs $O(n_{train}d)$; for a test set of size $n_{test}$, total is $O(n_{test}n_{train}d)$.

In [13]:
class KNNClassifier:
    """
    K-Nearest Neighbors classifier implemented from scratch with NumPy.

    Parameters
    ----------
    k : int, default=3
        Number of neighbors used for majority voting.
    metric : str, default='euclidean'
        Currently supports only Euclidean distance.
    """

    def __init__(self, k=3, metric="euclidean"):
        if k < 1:
            raise ValueError("k must be >= 1.")
        if metric != "euclidean":
            raise ValueError("Only Euclidean distance is supported in this implementation.")
        self.k = k
        self.metric = metric
        self.X_train = None
        self.y_train = None
        self.classes_ = None

    def fit(self, X_train, y_train):
        """Store training features and labels."""
        self.X_train = np.asarray(X_train, dtype=float)
        self.y_train = np.asarray(y_train)
        self.classes_ = np.unique(self.y_train)

        if self.k > self.X_train.shape[0]:
            raise ValueError("k cannot exceed number of training samples.")

        return self

    def _euclidean_distance_matrix(self, X_test):
        """Compute full pairwise Euclidean distance matrix in vectorized form."""
        return np.linalg.norm(X_test[:, None, :] - self.X_train[None, :, :], axis=2)

    def predict(self, X_test):
        """Predict class labels for test samples via majority voting."""
        if self.X_train is None or self.y_train is None:
            raise ValueError("Call fit before predict.")

        X_test = np.asarray(X_test, dtype=float)
        distances = self._euclidean_distance_matrix(X_test)

        # Get indices of k nearest neighbors for each test point.
        nearest_idx = np.argpartition(distances, kth=self.k - 1, axis=1)[:, : self.k]
        neighbor_labels = self.y_train[nearest_idx]

        # Vectorized majority vote via one-hot counting.
        class_idx = np.searchsorted(self.classes_, neighbor_labels)
        votes = np.eye(self.classes_.size, dtype=int)[class_idx].sum(axis=1)
        pred_idx = np.argmax(votes, axis=1)

        return self.classes_[pred_idx]

    def score(self, X_test, y_test):
        """Return mean classification accuracy on a test set."""
        y_pred = self.predict(X_test)
        y_test = np.asarray(y_test)
        return np.mean(y_pred == y_test)

## 8. Train-Test Split and Evaluation
We perform a manual 80%/20% train-test split using NumPy only (with stratification to preserve class balance). The K-NN algorithm is from scratch; scikit-learn is used only for dataset loading and evaluation metrics.

In [14]:
def manual_train_test_split(X, y, test_size=0.2, random_state=42, stratify=None):
    """
    Manually split arrays into train and test sets using NumPy.

    Parameters
    ----------
    X : np.ndarray of shape (n_samples, n_features)
        Feature matrix.
    y : np.ndarray of shape (n_samples,)
        Label vector.
    test_size : float or int, default=0.2
        If float, proportion of samples in test split. If int, absolute test count.
    random_state : int, default=42
        Seed for reproducibility.
    stratify : np.ndarray or None, default=None
        If provided, preserves class proportions approximately.

    Returns
    -------
    X_train, X_test, y_train, y_test : np.ndarray
        Split arrays.
    """
    X = np.asarray(X)
    y = np.asarray(y)

    if X.shape[0] != y.shape[0]:
        raise ValueError("X and y must have the same number of samples.")

    n_samples = X.shape[0]
    rng = np.random.default_rng(random_state)

    if isinstance(test_size, float):
        if not (0 < test_size < 1):
            raise ValueError("If test_size is float, it must be in (0, 1).")
        n_test_total = int(np.floor(test_size * n_samples))
    elif isinstance(test_size, int):
        if not (0 < test_size < n_samples):
            raise ValueError("If test_size is int, it must be in [1, n_samples - 1].")
        n_test_total = test_size
    else:
        raise TypeError("test_size must be float or int.")

    if stratify is None:
        perm = rng.permutation(n_samples)
        test_idx = perm[:n_test_total]
        train_idx = perm[n_test_total:]
    else:
        stratify = np.asarray(stratify)
        if stratify.shape[0] != n_samples:
            raise ValueError("stratify must have the same number of samples as X and y.")

        unique_labels = np.unique(stratify)
        test_parts = []
        train_parts = []

        for label in unique_labels:
            label_idx = np.where(stratify == label)[0]
            label_perm = rng.permutation(label_idx)

            if isinstance(test_size, float):
                n_label_test = int(np.floor(test_size * label_idx.size))
            else:
                n_label_test = int(np.round((label_idx.size / n_samples) * n_test_total))

            n_label_test = max(1, min(label_idx.size - 1, n_label_test))

            test_parts.append(label_perm[:n_label_test])
            train_parts.append(label_perm[n_label_test:])

        test_idx = np.concatenate(test_parts)
        train_idx = np.concatenate(train_parts)

        # Shuffle class-concatenated indices for a mixed ordering.
        test_idx = test_idx[rng.permutation(test_idx.size)]
        train_idx = train_idx[rng.permutation(train_idx.size)]

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = manual_train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Standardize using training statistics only to avoid data leakage.
train_mean = X_train.mean(axis=0)
train_std = np.where(X_train.std(axis=0) == 0, 1.0, X_train.std(axis=0))
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

knn = KNNClassifier(k=3)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"K-NN test accuracy (k=3): {acc:.4f}")

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=target_names,
    yticklabels=target_names
)
plt.title("K-NN Confusion Matrix (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

K-NN test accuracy (k=3): 0.9667


**Practical metric interpretation:** Accuracy is the fraction of correctly classified flowers on unseen test data. In the confusion matrix, diagonal entries are correct predictions, while off-diagonal entries show specific species confusions (typically between Versicolor and Virginica).

## 9. Assingment Questions

### Q1. What does the integer $k$ represent in K-Means versus K-NN?
- **K-Means:** $k$ is the number of clusters to discover in unlabeled data.
- **K-NN:** $k$ is the number of nearest labeled neighbors used to vote for a class label.

### Q2. How did the code structure reflect supervised vs. unsupervised learning?
- **Unsupervised (K-Means):** `kmeans` uses only `X` (no labels) and iteratively optimizes cluster assignments.
- **Supervised (K-NN):** `KNNClassifier.fit(X_train, y_train)` stores labeled data, and `predict` uses neighbors' labels for classification.

### Q3. When does K-Means do the heavy lifting?
The heavy lifting occurs during repeated distance-matrix computation and assignment updates in each iteration. Computationally, this dominates runtime at approximately $O(nkdI)$ over $I$ iterations.

### Q4. Briefly discuss scatter-plot discrepancies.
Differences between ground truth and cluster plots arise because K-Means optimizes geometric compactness, not label agreement. Also, 2D projections can hide separability present in 4D space, and overlapping species (especially Versicolor/Virginica) naturally create mismatches.

## 10. Conclusion and Possible Improvements
This notebook demonstrated from-scratch, vectorized implementations of K-Means and K-NN on Iris. K-Means found a reasonable 3-cluster structure, while K-NN achieved strong supervised performance. Key limitations include K-Means sensitivity to initialization and K-NN sensitivity to feature scaling and high-dimensional distance behavior. Improvements include full k-means++ diagnostics across multiple restarts, weighted K-NN voting, support for additional distance metrics, and explicit tie-breaking policies for reproducibility.

## 11. References and Acknowledgements
- Fisher, R. A. (1936). *The use of multiple measurements in taxonomic problems*.
- Scikit-learn documentation: Iris dataset (`sklearn.datasets.load_iris`) and evaluation utilities.
- Hastie, Tibshirani, and Friedman. *The Elements of Statistical Learning* (for clustering and nearest-neighbor concepts).